# PolicyRec v1.1.4 — 카테고리 / 중복공고 / 대표 main 구조 정리

## 이 노트북의 목적

**큰 부분**
- 3개 소스(`biz`, `kst`, `youth`)에서 각 200건씩 시도해 가져옵니다.
  (실제 합계는 568건 — youth API 가 168건만 반환)
- 카테고리 / scope / dedupe / main 구조를 한 번 정리하고
- 사람이 검토할 수 있는 csv 로 저장합니다.

**세세한 부분**
- 검색 고도화 / 대량 수집은 이번 단계의 목적이 아닙니다 (다음 단계).
- 노트북 = 흐름과 검토 중심. API 호출 / 파싱은 `app/`, `scripts/` 에 맡깁니다.
- 결과 csv 는 `data/csv/raw/`, `data/csv/main/`, `data/csv/rule/` 폴더에 역할별로 저장합니다.
- 다음 단계는 **supabase 적재 + Next.js 앱 연결** (별도 작업자) — 13. 인수인계 섹션 참고.

## 흐름 요약

1. 셋업 (경로 / 버전 변수)
2. 데이터 수집 실행 (`scripts/fetch.py`)
3. raw json 생성 확인
4. 정규화 실행 (`app/norm.py` 의 `save_combined_csv`) → raw csv 저장
5. raw csv 미리보기
6. 공통 컬럼 유지 + `raw_` 참고 컬럼 정리
7. `s_category` 규칙 적용
8. `_scope`, `_scope_reason` 규칙 적용
9. `norm_` 컬럼 생성 (비교용 정규화)
10. dedupe 후보 생성
11. 대표 `main` 행 만들기 + csv 저장
12. 결과 검토 / 다음 작업 메모
13. 인수인계 ← 이 노트북의 마지막 섹션

## 컬럼 접두어 규칙
- 접두어 없음 : 공통 핵심 컬럼 (예: `source`, `source_id`, `title`, `summary`, `provider`, `region`)
- `s_*`       : 서비스용으로 통합 / 정리한 컬럼 (예: `s_category`)
- `raw_*`     : 원본에서 거의 그대로 가져온 참고용 컬럼 (일부 source / 일부 행만 값이 있을 수 있음)
- `_*`        : 내부 처리용 (예: `_scope`, `_dedupe_key`)
- `norm_*`    : 비교용 정규화 (예: `norm_title`, `norm_period`)

## 0. 셋업

**큰 부분**
- 노트북 어디서 무엇을 저장할지 경로 변수를 한곳에 모읍니다.
- 이후 모든 셀은 이 변수만 사용합니다.

**세세한 부분**
- 폴더는 역할별로 고정 (`data/csv/raw`, `data/csv/main`, `data/csv/rule`)
- 파일명에는 버전 `v1_1_4`만 붙입니다.

In [54]:
from pathlib import Path
import sys, subprocess, json, re, html
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)

# 프로젝트 루트 (이 노트북이 있는 폴더)
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 현재 버전
VERSION = "v1_1_4"

# 출력 폴더 (역할 기준 고정)
CSV_ROOT     = PROJECT_ROOT / "data" / "csv"
CSV_RAW_DIR  = CSV_ROOT / "raw"
CSV_MAIN_DIR = CSV_ROOT / "main"
CSV_RULE_DIR = CSV_ROOT / "rule"
for d in [CSV_RAW_DIR, CSV_MAIN_DIR, CSV_RULE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 결과 파일 경로 (버전명만 다르게)
RAW_CSV       = CSV_RAW_DIR  / f"raw_{VERSION}.csv"
MAIN_CSV      = CSV_MAIN_DIR / f"main_{VERSION}.csv"
FLOW_CSV      = CSV_MAIN_DIR / f"flow_{VERSION}.csv"
CAT_RULE_CSV  = CSV_RULE_DIR / f"cat_rule_{VERSION}.csv"
SCOPE_RULE_CSV= CSV_RULE_DIR / f"scope_rule_{VERSION}.csv"

# 수집 목표
TARGET_PER_SOURCE = 200
SOURCES = ["biz", "kst", "youth"]

print("PROJECT_ROOT :", PROJECT_ROOT)
print("VERSION      :", VERSION)
print("RAW_CSV      :", RAW_CSV)
print("MAIN_CSV     :", MAIN_CSV)

PROJECT_ROOT : g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec
VERSION      : v1_1_4
RAW_CSV      : g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\raw\raw_v1_1_4.csv
MAIN_CSV     : g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\main\main_v1_1_4.csv


## 1. 데이터 수집 실행

**큰 부분**
- `scripts/fetch.py`를 한 번 호출하면 3개 소스 모두 수집됩니다.
- 한 source당 200건을 한 번의 호출로 받아옵니다 (page_size=200).

**세세한 부분**
- 같은 prefix의 raw json이 이미 있으면 자동으로 캐시 재사용 (`--force` 안 줌)
- 새로 받고 싶으면 아래 셀에서 `FORCE = True`로 바꿔서 실행합니다.

In [55]:
import os

FORCE = False  # True 로 바꾸면 캐시 무시하고 다시 호출

cmd = [
    sys.executable, "scripts/fetch.py",
    "--sources", *SOURCES,
    "--page", "1",
    "--page-size", str(TARGET_PER_SOURCE),
]
if FORCE:
    cmd.append("--force")

# 자식 프로세스 출력도 utf-8로 받게 환경변수 지정
child_env = os.environ.copy()
child_env["PYTHONIOENCODING"] = "utf-8"

print("실행:", " ".join(cmd))
result = subprocess.run(
    cmd, cwd=str(PROJECT_ROOT),
    capture_output=True, text=True,
    encoding="utf-8", errors="replace",
    env=child_env,
)
print(result.stdout[-1500:] if result.stdout else "(stdout 없음)")
if result.returncode != 0:
    print("ERR:", (result.stderr or "")[-500:])

실행: c:\Users\User\miniconda3\envs\Policyrec\python.exe scripts/fetch.py --sources biz kst youth --page 1 --page-size 200
PolicyRec 수집을 시작합니다.
실행 대상 source: biz, kst, youth
page=1, page_size=200, force=False

--- biz 수집 시작 ---
[Bizinfo] 기존 raw 파일이 있어서 API 호출을 건너뜁니다: G:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\raw\biz\bizinfo_page1_size200_20260427_185626.json

--- kst 수집 시작 ---
[K-Startup] 기존 raw 파일이 있어서 API 호출을 건너뜁니다: G:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\raw\kst\kstartup_page1_size200_20260427_185627.json

--- youth 수집 시작 ---
[Youthcenter] 기존 raw 파일이 있어서 API 호출을 건너뜁니다: G:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\raw\youth\youthcenter_page1_size200_20260427_185627.json

===== 수집 결과 요약 =====
- Bizinfo: 건너뜀
  설명: 기존 캐시 파일을 재사용했습니다.
  파일: G:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\raw\biz\bizinfo_page1_size200_20260427_185626.json
- K-Startup: 건너뜀
  설명: 기존 캐시 파일을 재사용했습니다.
  파일: G:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\raw\kst\kstartup_page1_size200_20260427_185627.json
- Youthcenter: 건너뜀
  설명: 기존 캐시 파일을 재사용했

## 2. raw json 생성 확인

**큰 부분**
- 각 source별로 가장 최근 json 파일과 그 안의 항목 수를 확인합니다.

**세세한 부분**
- 200 ± 약간의 오차면 정상
- 0건이거나 너무 적으면 1단계를 다시 봐야 합니다.

In [56]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"

def latest_raw(src):
    folder = RAW_DIR / src
    files = sorted(folder.glob("*.json"), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

def count_items(src, path):
    with path.open("r", encoding="utf-8") as f:
        d = json.load(f)
    if src == "biz":
        return len(d.get("jsonArray", []))
    if src == "kst":
        return len(d.get("data", []))
    if src == "youth":
        return len(d.get("result", {}).get("youthPolicyList", []))
    return 0

rows = []
for s in SOURCES:
    p = latest_raw(s)
    rows.append({"source": s, "file": p.name if p else None,
                 "count": count_items(s, p) if p else 0})
display(pd.DataFrame(rows))

,source,file,count
0,biz,bizinfo_page1_size200_20260427_185626.json,200
1,kst,kstartup_page1_size200_20260427_185627.json,200
2,youth,youthcenter_page1_size200_20260427_185627.json,200


## 3. 정규화 실행 → raw csv 저장

**큰 부분**
- `app/norm.py`의 `save_combined_csv`를 호출해 3개 소스의 최신 raw json 1개씩을 합쳐 csv로 저장합니다.
- 출력 경로를 `data/csv/raw/raw_v1_1_4.csv`로 지정합니다.
- 옆에 `*.meta.json`도 같이 저장 (생성 시간/raw 파일명/소스별 행수)

**세세한 부분**
- 이 단계의 컬럼은 기존 공통 스키마(`source`, `source_id`, `title` …) 그대로입니다.
- 다음 단계에서 `raw_` 접두어를 붙여 v1.1.4 컬럼 규칙으로 바꿉니다.

In [57]:
from app.norm import save_combined_csv

out = save_combined_csv(output_path=str(RAW_CSV), save_meta=True)
print("raw csv 저장:", out)

[norm] biz: using latest raw file bizinfo_page1_size200_20260427_185626.json
[norm] biz: normalized 200 rows from bizinfo_page1_size200_20260427_185626.json
[norm] kst: using latest raw file kstartup_page1_size200_20260427_185627.json
[norm] kst: normalized 200 rows from kstartup_page1_size200_20260427_185627.json
[norm] youth: using latest raw file youthcenter_page1_size200_20260427_185627.json
[norm] youth: normalized 200 rows from youthcenter_page1_size200_20260427_185627.json
[norm] before dedup: 600
[norm] removed duplicates: 0
[norm] after dedup: 600
[norm] saved combined csv: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\raw\raw_v1_1_4.csv
[norm] row count: 600
[norm] saved meta: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\raw\raw_v1_1_4.meta.json
raw csv 저장: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\raw\raw_v1_1_4.csv


## 4. raw csv 미리보기

**큰 부분**
- 방금 만든 csv를 읽어 모양과 source별 행수를 확인합니다.

**세세한 부분**
- shape이 `(600, 22)`면 정상 (소스당 200건 × 22컬럼)

In [58]:
df_raw = pd.read_csv(RAW_CSV, dtype=str).fillna("")
print("shape:", df_raw.shape)
display(df_raw["source"].value_counts().rename_axis("source").reset_index(name="count"))
display(df_raw[["source","source_id","title","category","region"]].head(3))

shape: (600, 22)


,source,count
0,biz,200
1,kst,200
2,youth,200


,source,source_id,title,category,region
0,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,기술,기후에너지환경부
1,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,기술,경상남도
2,biz,PBLN_000000000121346,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,내수,경기도


## 5. 공통 컬럼 유지 + raw_ 참고 컬럼 정리

**큰 부분**
- 공통 핵심 컬럼은 접두어 없이 유지합니다.
- 일부 source 또는 일부 행에만 값이 있는 참고용 컬럼만 `raw_` 접두어로 둡니다.

**세세한 부분**
- `source`, `source_id`, `title`, `summary`, `region`은 그대로 둡니다.
- `provider`는 운영기관 우선, 비어 있으면 주관기관을 쓰는 대표값으로 둡니다.
- `raw_period_text`는 원문 그대로 (`raw_apply_start ~ raw_apply_end` 형태로 합칩니다).

In [59]:
rename_map = {
    "category": "raw_category",
    "subcategory": "raw_subcategory",
    "supervising_agency": "raw_supervising_agency",
    "operating_agency": "raw_operating_agency",
    "target_group": "raw_target_group",
    "target_age": "raw_target_age",
    "target_detail": "raw_target_detail",
    "income_condition": "raw_income_condition",
    "startup_stage": "raw_startup_stage",
    "support_type": "raw_support_type",
    "application_method": "raw_application_method",
    "required_documents": "raw_required_documents",
    "additional_conditions": "raw_additional_conditions",
    "apply_start": "raw_apply_start",
    "apply_end": "raw_apply_end",
    "detail_url": "raw_detail_url",
}
df = df_raw.rename(columns=rename_map).copy()

# 대표 기관명은 운영기관 우선, 비어 있으면 주관기관으로 보강합니다.
df["provider"] = df["raw_operating_agency"].where(
    df["raw_operating_agency"].astype(str).str.strip() != "",
    df["raw_supervising_agency"],
)

# 원문 기간 텍스트는 보존용으로 두 값을 합쳐서 따로 둡니다.
df["raw_period_text"] = (df["raw_apply_start"].fillna("") + " ~ " + df["raw_apply_end"].fillna("")).str.strip(" ~")

print("shape:", df.shape)
print("컬럼 일부:", [c for c in df.columns if c.startswith("raw_")][:10])

shape: (600, 24)
컬럼 일부: ['raw_category', 'raw_subcategory', 'raw_supervising_agency', 'raw_operating_agency', 'raw_target_group', 'raw_target_age', 'raw_target_detail', 'raw_income_condition', 'raw_startup_stage', 'raw_support_type']


## 6. `s_category` 규칙 적용

**큰 부분**
- source마다 `raw_category`가 다른 표현으로 되어 있어, 통일된 한국어 라벨(`s_category`)로 매핑합니다.
- 매핑 규칙은 dict로 정의하고, 보기 좋게 csv로도 저장합니다.

**세세한 부분**
- 매핑되지 않은 값은 `기타`로 둡니다 (룰을 검토하면서 점차 보완).
- `kst`의 `R&amp;D` 같은 HTML entity는 미리 풀어줍니다.

In [60]:
# (source, raw_category) → s_category 한국어 라벨
CAT_RULES = [
    # biz
    ("biz", "창업",     "창업"),
    ("biz", "경영",     "경영"),
    ("biz", "기술",     "기술"),
    ("biz", "인력",     "인력/일자리"),
    ("biz", "수출",     "판로/수출"),
    ("biz", "내수",     "판로/수출"),
    ("biz", "금융",     "자금"),
    ("biz", "기타",     "기타"),
    # kst
    ("kst", "사업화",                       "창업"),
    ("kst", "창업교육",                     "창업"),
    ("kst", "멘토링ㆍ컨설팅ㆍ교육",          "교육/멘토링"),
    ("kst", "시설ㆍ공간ㆍ보육",              "시설/공간"),
    ("kst", "행사ㆍ네트워크",                "행사/네트워크"),
    ("kst", "판로ㆍ해외진출",                "판로/수출"),
    ("kst", "글로벌",                       "판로/수출"),
    ("kst", "기술개발(R&D)",                 "기술"),
    ("kst", "인력",                         "인력/일자리"),
    ("kst", "정책자금",                     "자금"),
    # youth
    ("youth", "일자리",         "인력/일자리"),
    ("youth", "주거",           "주거"),
    ("youth", "교육･직업훈련",   "교육/멘토링"),
    ("youth", "금융･복지･문화",  "복지/문화"),
    ("youth", "참여･기반",      "참여/기반"),
]

cat_rule_df = pd.DataFrame(CAT_RULES, columns=["source","raw_category","s_category"])
cat_rule_df.to_csv(CAT_RULE_CSV, index=False, encoding="utf-8-sig")
print("카테고리 규칙 저장:", CAT_RULE_CSV)
display(cat_rule_df.head(15))

카테고리 규칙 저장: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\rule\cat_rule_v1_1_4.csv


,source,raw_category,s_category
0,biz,창업,창업
1,biz,경영,경영
2,biz,기술,기술
3,biz,인력,인력/일자리
4,biz,수출,판로/수출
5,biz,내수,판로/수출
6,biz,금융,자금
7,biz,기타,기타
8,kst,사업화,창업
9,kst,창업교육,창업


In [61]:
# raw_category 정리 (HTML entity 풀기, ',' 중복 제거 등)
def _clean_raw_cat(v):
    if not isinstance(v, str):
        return ""
    v = html.unescape(v).strip()
    # youth에서 자주 보이는 "일자리,일자리" 같은 중복을 1개로
    parts = [p.strip() for p in v.split(",") if p.strip()]
    if parts and all(p == parts[0] for p in parts):
        return parts[0]
    return v

df["raw_category_clean"] = df["raw_category"].apply(_clean_raw_cat)

rule_lookup = {(r.source, r.raw_category): r.s_category for r in cat_rule_df.itertuples(index=False)}
df["s_category"] = df.apply(
    lambda r: rule_lookup.get((r["source"], r["raw_category_clean"]), "기타"),
    axis=1,
)

print("s_category 분포:")
display(df["s_category"].value_counts().rename_axis("s_category").reset_index(name="count"))

print("\n매핑 안 된 (raw_category → 기타) 상위:")
miss = df[df["s_category"]=="기타"][["source","raw_category_clean"]].value_counts().head(10).rename_axis(["source","raw_category_clean"]).reset_index(name="count")
display(miss)

s_category 분포:


,s_category,count
0,창업,94
1,인력/일자리,86
2,판로/수출,79
3,경영,64
4,교육/멘토링,61
5,복지/문화,60
6,기술,43
7,참여/기반,34
8,시설/공간,29
9,주거,23



매핑 안 된 (raw_category → 기타) 상위:


,source,raw_category_clean,count
0,biz,기타,2


## 7. `_scope`, `_scope_reason` 규칙 적용

**큰 부분**
- `_scope`는 "이 행을 서비스에 보여줄지" 결정하는 내부 라벨입니다.
- 이번 단계에서는 두 값만 사용:
  - `main`: 대표 행 (서비스에 보여줄 후보)
  - `dup`: 같은 dedupe 그룹의 추가 사본

**세세한 부분**
- 1차 단계에서는 모든 행을 `main`으로 두고, dedupe 단계에서 `dup`을 따로 표시합니다.
- `_scope_reason`은 왜 그 scope가 됐는지 짧은 사유.

In [62]:
SCOPE_RULES = [
    {"scope": "main",  "의미": "대표 공고. 서비스 노출 후보"},
    {"scope": "other", "의미": "main 외. 자세한 사유는 _scope_reason 참조"},
]
scope_rule_df = pd.DataFrame(SCOPE_RULES)
scope_rule_df.to_csv(SCOPE_RULE_CSV, index=False, encoding="utf-8-sig")
print("scope 규칙 저장:", SCOPE_RULE_CSV)
display(scope_rule_df)

# 1차 부여: 모든 행 main
df["_scope"] = "main"
df["_scope_reason"] = "primary"
print()
print("초기 _scope 분포:", df["_scope"].value_counts().to_dict())

scope 규칙 저장: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\rule\scope_rule_v1_1_4.csv


,scope,의미
0,main,대표 공고. 서비스 노출 후보
1,other,main 외. 자세한 사유는 _scope_reason 참조



초기 _scope 분포: {'main': 600}


## 8. `norm_` 컬럼 생성 (비교용 정규화)

**큰 부분**
- 같은 공고를 다른 source에서 받았을 때 비교하기 쉽게, 비교 전용 정규화 컬럼을 만듭니다.
- 원문은 `raw_*`에 그대로 보존되어 있습니다.

**세세한 부분**
- `norm_title`: 공백·특수기호·괄호·HTML entity 제거, 소문자
- `norm_provider`: 운영기관명에서 (재)/(주) 등 형식어 제거
- `norm_period`: `start~end` 합쳐 숫자만 남김
- `norm_detail_url`: scheme/쿼리 제거 후 lowercase

In [63]:
_RX_NON = re.compile(r"[\s\W_]+", flags=re.UNICODE)
_RX_PAREN = re.compile(r"\([^)]*\)")
_RX_DIGITS = re.compile(r"\d+")
_RX_SCHEME = re.compile(r"^https?://", flags=re.IGNORECASE)
_RX_FRAG = re.compile(r"#.*$")
# R&D 변형들을 한 토큰("rnd")으로 통일 → 표기 차이가 dedupe에 묻히지 않게
_RX_RND = re.compile(r"R\s*&\s*D|R\s*and\s*D|R\s*amp\s*D|RnD|R＆D", flags=re.IGNORECASE)

def norm_text(v):
    if not isinstance(v, str): return ""
    v = html.unescape(v)
    v = _RX_RND.sub("rnd", v)        # R&D 류 표준화 (소문자 rnd)
    v = _RX_PAREN.sub("", v)
    v = _RX_NON.sub("", v)
    return v.lower()

def norm_url(v):
    # query(?param=...)는 보존: 공고 식별자가 query에 있는 경우가 많음.
    # fragment(#...)만 제거하고 scheme/마지막 슬래시만 정리.
    if not isinstance(v, str): return ""
    v = v.strip()
    v = _RX_SCHEME.sub("", v)
    v = _RX_FRAG.sub("", v)
    return v.lower().rstrip("/")

def norm_period(start, end):
    s = "".join(_RX_DIGITS.findall(str(start or "")))
    e = "".join(_RX_DIGITS.findall(str(end or "")))
    return f"{s}~{e}" if (s or e) else ""

df["norm_title"]      = df["title"].apply(norm_text)
df["norm_provider"]   = df["provider"].apply(norm_text)
df["norm_period"]     = df.apply(lambda r: norm_period(r["raw_apply_start"], r["raw_apply_end"]), axis=1)
df["norm_detail_url"] = df["raw_detail_url"].apply(norm_url)

display(df[["source","title","provider","norm_title","norm_provider","norm_period","norm_detail_url"]].head(5))

,source,title,provider,norm_title,norm_provider,norm_period,norm_detail_url
0,biz,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,한국에너지기술평가원,2026년한체코ㆍ한중국에너지국제공동rnd신규지원대상과제공고,한국에너지기술평가원,,www.bizinfo.go.kr/sii/siia/selectsiia200detail...
1,biz,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,한국생산기술연구원,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,20260423~20260507,www.bizinfo.go.kr/sii/siia/selectsiia200detail...
2,biz,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,경기테크노파크,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고,경기테크노파크,20260423~20260522,www.bizinfo.go.kr/sii/siia/selectsiia200detail...
3,biz,[경기] 2026년 가상융합 사업화지원 (TRACK 2 : 도쿄 XR페어 참가 지원...,경기콘텐츠진흥원,경기2026년가상융합사업화지원연장공고,경기콘텐츠진흥원,20260424~20260430,www.bizinfo.go.kr/sii/siia/selectsiia200detail...
4,biz,2026년 패션 디자이너 브랜드 스케일업 지원 참가사 모집 공고,한국콘텐츠진흥원,2026년패션디자이너브랜드스케일업지원참가사모집공고,한국콘텐츠진흥원,20260423~20260514,www.bizinfo.go.kr/sii/siia/selectsiia200detail...


## 9. dedupe 후보 생성

**큰 부분**
- 두 가지 신호로 중복 후보를 잡습니다.
  - **강한 신호**: `norm_detail_url`이 정확히 같음 (다른 source끼리도 잡힘)
  - **약한 신호**: `_dedupe_key = norm_title + norm_provider + norm_period`이 같음
- 자동으로 합치지는 않습니다. 후보로만 표시합니다 (검토 후 합치게).

**세세한 부분**
- 강한 신호로 묶인 그룹은 첫 행을 `main`, 나머지를 `dup`으로 표시.
- 약한 신호 후보는 `_dup_candidate=True` 컬럼으로만 표시 (scope 변경 X).

In [64]:
df["_dedupe_key"] = df["norm_title"] + "|" + df["norm_provider"] + "|" + df["norm_period"]

# --- 강한 신호: norm_detail_url 정확 일치 (빈 값은 제외) ---
url_groups = df[df["norm_detail_url"].astype(str).str.len() > 0].groupby("norm_detail_url")
url_dup_groups = [g for _, g in url_groups if len(g) > 1]
print(f"강한 신호(URL exact) 그룹 수: {len(url_dup_groups)}")

for g in url_dup_groups:
    keep_idx = g.index[0]
    drop_idx = g.index[1:]
    df.loc[drop_idx, "_scope"] = "other"
    df.loc[drop_idx, "_scope_reason"] = f"url-exact-dup-of:{g.loc[keep_idx,'source_id']}"

# --- 약한 신호: _dedupe_key 일치 (빈 키 제외, URL 강한 신호로 이미 other된 것 제외) ---
df["_dup_candidate"] = False
weak = df[(df["_scope"]=="main") & (df["_dedupe_key"].str.replace("|","",regex=False).str.len()>0)]
weak_groups = weak.groupby("_dedupe_key")
weak_dup_groups = [g for _, g in weak_groups if len(g) > 1]
print(f"약한 신호(_dedupe_key) 후보 그룹 수: {len(weak_dup_groups)}")

for g in weak_dup_groups:
    df.loc[g.index, "_dup_candidate"] = True

print()
print("_scope 분포(강한 신호 반영 후):")
display(df["_scope"].value_counts().rename_axis("_scope").reset_index(name="count"))

print()
print("_dup_candidate 분포(약한 신호):")
display(df["_dup_candidate"].value_counts().rename_axis("_dup_candidate").reset_index(name="count"))

강한 신호(URL exact) 그룹 수: 17
약한 신호(_dedupe_key) 후보 그룹 수: 11

_scope 분포(강한 신호 반영 후):


,_scope,count
0,main,568
1,other,32



_dup_candidate 분포(약한 신호):


,_dup_candidate,count
0,False,578
1,True,22


In [65]:
# 약한 신호 후보 그룹 예시 (상위 5개 그룹만 미리보기)
if weak_dup_groups:
    print("=== 약한 신호 후보 그룹 예시 ===")
    for g in weak_dup_groups[:5]:
        print(f"\n[group key={g['_dedupe_key'].iloc[0][:60]}...]")
        display(g[["source","source_id","title","provider","raw_apply_start","raw_apply_end"]])
else:
    print("약한 신호 후보 그룹 없음.")

=== 약한 신호 후보 그룹 예시 ===

[group key=2026년더현대글로벌더셀렉츠글로벌팝업스토어참가사모집공고|한국콘텐츠진흥원|20260420~20260511...]


,source,source_id,title,provider,raw_apply_start,raw_apply_end
80,biz,PBLN_000000000121263,2026년 더현대글로벌×더셀렉츠 글로벌 팝업스토어(프랑스 파리) 참가사 모집 공고,한국콘텐츠진흥원,2026-04-20,2026-05-11
82,biz,PBLN_000000000121261,2026년 더현대글로벌×더셀렉츠 글로벌 팝업스토어(일본 도쿄) 참가사 모집 공고,한국콘텐츠진흥원,2026-04-20,2026-05-11



[group key=2026년콘텐츠스타트업해외마켓참가기업추가모집공고|한국콘텐츠진흥원|20260417~20260511...]


,source,source_id,title,provider,raw_apply_start,raw_apply_end
165,biz,PBLN_000000000121174,2026년 콘텐츠 스타트업 해외마켓(SWITCH) 참가기업 추가모집 공고,한국콘텐츠진흥원,2026-04-17,2026-05-11
166,biz,PBLN_000000000121173,2026년 콘텐츠 스타트업 해외마켓(Ai4) 참가기업 추가모집 공고,한국콘텐츠진흥원,2026-04-17,2026-05-11



[group key=2026년하반기독립예술영화제작지원사업추가접수공고|영화진흥위원회|20260421~20260506...]


,source,source_id,title,provider,raw_apply_start,raw_apply_end
41,biz,PBLN_000000000121303,2026년 하반기 독립예술영화 제작지원(장편 극영화 부문) 사업 추가 접수 공고,영화진흥위원회,2026-04-21,2026-05-06
73,biz,PBLN_000000000121270,2026년 하반기 독립예술영화 제작지원(다큐멘터리 (제작ㆍ후반제작)) 사업 추가 접...,영화진흥위원회,2026-04-21,2026-05-06



[group key=광주2026년노동자휴게시설설치지원사업참여기업추가모집공고|광주경제진흥상생일자리재단|20260417~202604...]


,source,source_id,title,provider,raw_apply_start,raw_apply_end
112,biz,PBLN_000000000121229,[광주] 2026년 노동자 휴게시설 설치 지원사업 참여기업 추가 모집 공고,광주경제진흥상생일자리재단,2026-04-17,2026-04-30
124,biz,PBLN_000000000121215,[광주] 2026년 노동자 휴게시설 설치 지원사업 참여기업 추가 모집 공고,광주경제진흥상생일자리재단,2026-04-17,2026-04-30



[group key=광주2026년지역기업성장사다리지원사업통합공고|광주테크노파크|20260422~20260508...]


,source,source_id,title,provider,raw_apply_start,raw_apply_end
12,biz,PBLN_000000000121335,[광주] 2026년 지역기업 성장사다리 지원사업 통합 공고(일반기업),광주테크노파크,2026-04-22,2026-05-08
15,biz,PBLN_000000000121332,[광주] 2026년 지역기업 성장사다리 지원사업 통합 공고(지정기업),광주테크노파크,2026-04-22,2026-05-08


## 10. 대표 `main` 행 만들기 + csv 저장

**큰 부분**
- `_scope == 'main'` 행만 모아 `main_v1_1_4.csv`로 저장합니다.
- 동시에 흐름 단계별 행수를 `flow_v1_1_4.csv`로 같이 저장합니다.

**세세한 부분**
- main csv에는 서비스 표시·필터에 쓰일 컬럼 + 일부 raw 보존 컬럼만 남깁니다.
- 전체 raw는 `raw_v1_1_4.csv`에 그대로 있으므로 main에서는 정보 압축.

In [66]:
main_df = df[df["_scope"]=="main"].copy()

# 서비스용 표시 컬럼
main_df["title"]    = main_df["title"].apply(lambda v: html.unescape(str(v)).strip())
main_df["provider"] = main_df["provider"].apply(lambda v: html.unescape(str(v)).strip())
main_df["region"]   = main_df["region"].apply(lambda v: html.unescape(str(v)).strip())
main_df["summary"]  = main_df["summary"].apply(lambda v: html.unescape(str(v)).strip())

MAIN_COLS = [
    "source", "source_id",
    "title", "summary", "s_category", "provider", "region",
    "raw_target_group", "raw_support_type", "raw_target_age", "raw_income_condition",
    "raw_startup_stage", "raw_additional_conditions", "raw_required_documents",
    "raw_application_method",
    "raw_period_text",
    "_scope", "_scope_reason", "_dup_candidate",
    "_dedupe_key",
    "norm_title", "norm_provider", "norm_period", "norm_detail_url",
    "raw_detail_url",
]
main_out = main_df[MAIN_COLS].copy()
main_out.to_csv(MAIN_CSV, index=False, encoding="utf-8-sig")
print("main csv 저장:", MAIN_CSV)
print("main shape:", main_out.shape)
display(main_out.head(5))

main csv 저장: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\main\main_v1_1_4.csv
main shape: (568, 25)


,source,source_id,title,summary,s_category,provider,region,raw_target_group,raw_support_type,raw_target_age,...,raw_period_text,_scope,_scope_reason,_dup_candidate,_dedupe_key,norm_title,norm_provider,norm_period,norm_detail_url,raw_detail_url
0,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,"<p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제...",기술,한국에너지기술평가원,기후에너지환경부,중소기업,공동기술개발,,...,사업별 상이,main,primary,False,2026년한체코ㆍ한중국에너지국제공동rnd신규지원대상과제공고|한국에너지기술평가원|,2026년한체코ㆍ한중국에너지국제공동rnd신규지원대상과제공고,한국에너지기술평가원,,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
1,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,<p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련...,기술,한국생산기술연구원,경상남도,중소기업,기술사업화/이전/지도,,...,2026-04-23 ~ 2026-05-07,main,primary,False,경남2026년소재부품성장잠재기업육성사업수요기업모집공고|한국생산기술연구원|202604...,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,20260423~20260507,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
2,biz,PBLN_000000000121346,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,<p>이천시와 경기테크노파크에서는 관내 도ㆍ공예기업 우수제품의 온라인 매출 증대와 ...,판로/수출,경기테크노파크,경기도,중소기업,홍보지원,,...,2026-04-23 ~ 2026-05-22,main,primary,False,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고|경기테크노파크|...,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고,경기테크노파크,20260423~20260522,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
3,biz,PBLN_000000000121345,[경기] 2026년 가상융합 사업화지원 (TRACK 2 : 도쿄 XR페어 참가 지원...,<p>경기콘텐츠진흥원은 아래와 같이 ‘2026년 가상융합 기업 사업화지원’에 참여할...,경영,경기콘텐츠진흥원,경기도,중소기업,디자인/상품화/사업화,,...,2026-04-24 ~ 2026-04-30,main,primary,False,경기2026년가상융합사업화지원연장공고|경기콘텐츠진흥원|20260424~20260430,경기2026년가상융합사업화지원연장공고,경기콘텐츠진흥원,20260424~20260430,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
4,biz,PBLN_000000000121344,2026년 패션 디자이너 브랜드 스케일업 지원 참가사 모집 공고,"<p>패션 콘텐츠 제작, 홍보ㆍ마케팅 지원을 통한 성장기 디자이너 브랜드 인지도 제...",경영,한국콘텐츠진흥원,문화체육관광부,중소기업,디자인/상품화/사업화,,...,2026-04-23 ~ 2026-05-14,main,primary,False,2026년패션디자이너브랜드스케일업지원참가사모집공고|한국콘텐츠진흥원|20260423~...,2026년패션디자이너브랜드스케일업지원참가사모집공고,한국콘텐츠진흥원,20260423~20260514,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...


In [67]:
# 흐름 요약
flow_rows = [
    {"단계": "0_raw",            "행수": len(df_raw),                       "설명": "3개 소스 통합 raw csv"},
    {"단계": "1_after_category", "행수": len(df),                            "설명": "s_category 라벨 부여"},
    {"단계": "2_main",           "행수": int((df['_scope']=='main').sum()),  "설명": "main scope 행수 (강한 신호 dedupe 반영)"},
    {"단계": "3_other_url",      "행수": int((df['_scope']=='other').sum()), "설명": "URL exact match로 other 처리된 행수"},
    {"단계": "4_dup_weak_cand",  "행수": int(df['_dup_candidate'].sum()),     "설명": "_dedupe_key 약한 후보 행수 (검토 대상)"},
    {"단계": "5_main_final",     "행수": len(main_out),                      "설명": "main csv 최종 행수"},
]
flow_df = pd.DataFrame(flow_rows)
flow_df.to_csv(FLOW_CSV, index=False, encoding="utf-8-sig")
print("flow csv 저장:", FLOW_CSV)
display(flow_df)

flow csv 저장: g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\main\flow_v1_1_4.csv


,단계,행수,설명
0,0_raw,600,3개 소스 통합 raw csv
1,1_after_category,600,s_category 라벨 부여
2,2_main,568,main scope 행수 (강한 신호 dedupe 반영)
3,3_other_url,32,URL exact match로 other 처리된 행수
4,4_dup_weak_cand,22,_dedupe_key 약한 후보 행수 (검토 대상)
5,5_main_final,568,main csv 최종 행수


## 11. 결과 검토

**큰 부분**
- s_category × source 분포
- main csv 미리보기
- 약한 dedupe 후보 그룹 수

**세세한 부분**
- 다음 작업에서 사용자가 검토할 포인트:
  - 매핑 안 된(`기타`) raw_category 가 있는지
  - 약한 후보 중 진짜 같은 공고가 얼마나 되는지

In [68]:
# s_category × source 교차표
ct = pd.crosstab(df["s_category"], df["source"], margins=True, margins_name="합계")
print("=== s_category × source ===")
display(ct)

# main 미리보기
print("\n=== main csv 미리보기 ===")
display(main_out.head(5))

# 저장된 파일 목록
print("\n=== 이번 실행으로 저장된 csv ===")
for p in [RAW_CSV, MAIN_CSV, FLOW_CSV, CAT_RULE_CSV, SCOPE_RULE_CSV]:
    print(" -", p, "(", p.stat().st_size, "bytes )")

=== s_category × source ===


source,biz,kst,youth,합계
s_category,,,,
경영,64,0,0,64
교육/멘토링,0,43,18,61
기술,39,4,0,43
기타,2,0,0,2
복지/문화,0,0,60,60
시설/공간,0,29,0,29
인력/일자리,19,2,65,86
자금,8,1,0,9
주거,0,0,23,23



=== main csv 미리보기 ===


,source,source_id,title,summary,s_category,provider,region,raw_target_group,raw_support_type,raw_target_age,...,raw_period_text,_scope,_scope_reason,_dup_candidate,_dedupe_key,norm_title,norm_provider,norm_period,norm_detail_url,raw_detail_url
0,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,"<p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제...",기술,한국에너지기술평가원,기후에너지환경부,중소기업,공동기술개발,,...,사업별 상이,main,primary,False,2026년한체코ㆍ한중국에너지국제공동rnd신규지원대상과제공고|한국에너지기술평가원|,2026년한체코ㆍ한중국에너지국제공동rnd신규지원대상과제공고,한국에너지기술평가원,,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
1,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,<p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련...,기술,한국생산기술연구원,경상남도,중소기업,기술사업화/이전/지도,,...,2026-04-23 ~ 2026-05-07,main,primary,False,경남2026년소재부품성장잠재기업육성사업수요기업모집공고|한국생산기술연구원|202604...,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,20260423~20260507,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
2,biz,PBLN_000000000121346,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,<p>이천시와 경기테크노파크에서는 관내 도ㆍ공예기업 우수제품의 온라인 매출 증대와 ...,판로/수출,경기테크노파크,경기도,중소기업,홍보지원,,...,2026-04-23 ~ 2026-05-22,main,primary,False,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고|경기테크노파크|...,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고,경기테크노파크,20260423~20260522,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
3,biz,PBLN_000000000121345,[경기] 2026년 가상융합 사업화지원 (TRACK 2 : 도쿄 XR페어 참가 지원...,<p>경기콘텐츠진흥원은 아래와 같이 ‘2026년 가상융합 기업 사업화지원’에 참여할...,경영,경기콘텐츠진흥원,경기도,중소기업,디자인/상품화/사업화,,...,2026-04-24 ~ 2026-04-30,main,primary,False,경기2026년가상융합사업화지원연장공고|경기콘텐츠진흥원|20260424~20260430,경기2026년가상융합사업화지원연장공고,경기콘텐츠진흥원,20260424~20260430,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...
4,biz,PBLN_000000000121344,2026년 패션 디자이너 브랜드 스케일업 지원 참가사 모집 공고,"<p>패션 콘텐츠 제작, 홍보ㆍ마케팅 지원을 통한 성장기 디자이너 브랜드 인지도 제...",경영,한국콘텐츠진흥원,문화체육관광부,중소기업,디자인/상품화/사업화,,...,2026-04-23 ~ 2026-05-14,main,primary,False,2026년패션디자이너브랜드스케일업지원참가사모집공고|한국콘텐츠진흥원|20260423~...,2026년패션디자이너브랜드스케일업지원참가사모집공고,한국콘텐츠진흥원,20260423~20260514,www.bizinfo.go.kr/sii/siia/selectsiia200detail...,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...



=== 이번 실행으로 저장된 csv ===
 - g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\raw\raw_v1_1_4.csv ( 2399570 bytes )
 - g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\main\main_v1_1_4.csv ( 854139 bytes )
 - g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\main\flow_v1_1_4.csv ( 338 bytes )
 - g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\rule\cat_rule_v1_1_4.csv ( 740 bytes )
 - g:\다른 컴퓨터\내 컴퓨터\github\PolicyRec\data\csv\rule\scope_rule_v1_1_4.csv ( 120 bytes )


## 12. 다음 작업 메모

**이번 단계에서 정리한 것**
- 600건 raw 데이터 + 통합 raw csv
- 카테고리 매핑 규칙(`cat_rule_v1_1_4.csv`)
- scope 규칙(`scope_rule_v1_1_4.csv`)
- dedupe 후보(강한 신호 / 약한 신호)
- 대표 main 행(`main_v1_1_4.csv`)
- 흐름 요약(`flow_v1_1_4.csv`)

**검토하면 좋은 것**
- `s_category`가 `기타`로 빠진 raw 값들을 보고 규칙을 보강
- 약한 dedupe 후보 그룹 중 실제로 같은 공고를 어떻게 처리할지
- main csv를 streamlit 쪽에서 어떻게 연결할지

**다음 노트북 예정 (별도 작업)**
- `PolicyRec_v1_2_3.ipynb` — BM25/벡터/DB 활용 검색 축이 바뀔 때

## 13. 인수인계 (다음 사람을 위한 메모)

**큰 부분**
- 이 노트북은 **200건 PoC** 기준입니다.
- 컬럼 의미 / 채움률 / vector text 재료 / 메타 정보 / 팀원 공유 메시지를 한곳에 모아둡니다.
- 다음 단계 담당 (노트북 정제 · supabase DB · Next.js 앱) 이 이 섹션만 읽어도 인계가 되도록 정리합니다.

**세세한 부분**
- main csv 컬럼은 **더 늘리지 않습니다.** (룰 버전은 별도 csv + `*.meta.json` 으로 추적)
- vector text 는 main 기준으로 만들고, supabase 적재 시 별도 vector 컬럼으로 사용합니다.
- 앱은 streamlit 이 아니라 **Next.js**. `streamlit_app3*.py` 는 PoC 참고용입니다.

### 이 섹션의 셀 구성

| 번호 | 내용 |
| --- | --- |
| 13-1 | 컬럼별 source 별 채움률 표 |
| 13-2 | 컬럼 역할 분류 (공통 필터 / vector text 재료 / 내부용 등) |
| 13-3 | `build_vector_text(row)` 빌더 함수 + 샘플 |
| 13-4 | `main_v1_1_4.meta.json` 저장 |
| 13-5 | 다음 사람 (정제 / supabase / Next.js) 에게 전할 메시지 |


In [69]:
# ======================================================================
# 13-1. 컬럼 dictionary + source 별 채움률
# ======================================================================
# 큰 부분 :
#   main csv 의 각 컬럼이 source 별로 얼마나 채워져 있는지 한 표로 봅니다.
#
# 세세한 부분 :
#   - 채움률이 source 마다 크게 다른 컬럼은 "공통 필터" 로 쓸 수 없습니다.
#   - 그런 컬럼은 사람 검토용 / vector text 재료로만 활용합니다.
#   - 이 표는 검토자 / DB 담당에게 공유하면 좋습니다.

# main csv 다시 읽기 (이 셀만 따로 실행해도 동작하게 모든 값을 문자열로 읽습니다)
main_df_review = pd.read_csv(MAIN_CSV, dtype=str, keep_default_na=False)

# 데이터에 실제로 들어 있는 source 목록 (정렬해서 안정된 순서로)
sources_in_data = sorted(main_df_review["source"].unique().tolist())

# 컬럼별로 source 별 "비어있지 않은 행 비율(%)" 계산
fill_rows = []
for col in main_df_review.columns:
    row = {"column": col}
    for src in sources_in_data:
        sub = main_df_review[main_df_review["source"] == src]
        # 공백/빈문자열은 비어있는 것으로 간주
        nonblank = sub[col].astype(str).str.strip().ne("").sum()
        pct = round(nonblank / len(sub) * 100, 1) if len(sub) else 0.0
        row[f"{src}_fill%"] = pct
    fill_rows.append(row)

fill_df = pd.DataFrame(fill_rows)
print("=== 컬럼별 채움률 (source 별, 단위 %) ===")
display(fill_df)


# ======================================================================
# 13-2. 컬럼 역할 분류 (공통 필터 vs vector text 재료 vs 내부용)
# ======================================================================
# 큰 부분 :
#   각 컬럼이 어떤 역할을 하는지를 한 줄씩 정리합니다.
#
# 세세한 부분 :
#   - "공통 필터"   : 카드 / 검색 화면의 메인 필터로 노출할 컬럼
#   - "vector text 재료" : 임베딩 텍스트에 합쳐질 컬럼 (sparse 해도 OK)
#   - "식별자"      : (source, source_id) 같은 PK 후보
#   - "비교 정규화용" : norm_* 류 — dedupe / 매칭 비교용
#   - "내부 처리용"  : _ 로 시작 — _scope, _dedupe_key 등
#   - "참고용"      : 위에 해당하지 않는 raw_* 컬럼 (사람 검토 / vector 재료)

# 메인 필터로 실제로 노출할 컬럼 (사람이 정한 3개)
SERVICE_FILTERS = ["s_category", "provider", "region"]

# vector text 에 들어갈 후보 컬럼들
VECTOR_TEXT_COLS = [
    "title", "summary", "s_category", "provider", "region",
    "raw_target_group", "raw_support_type", "raw_period_text",
    "raw_target_age", "raw_income_condition", "raw_startup_stage",
    "raw_additional_conditions", "raw_required_documents",
    "raw_application_method",
]


def classify_column(col: str) -> str:
    """컬럼 이름을 받아서 어떤 역할인지 라벨을 돌려줍니다."""
    if col in ("source", "source_id"):
        return "식별자"
    if col in SERVICE_FILTERS:
        return "공통 필터"
    if col in VECTOR_TEXT_COLS:
        return "vector text 재료"
    if col.startswith("norm_"):
        return "비교 정규화용"
    if col.startswith("_"):
        return "내부 처리용"
    if col.startswith("raw_"):
        return "참고용 (raw)"
    return "기타"


classify_df = pd.DataFrame(
    {"column": main_df_review.columns,
     "역할": [classify_column(c) for c in main_df_review.columns]}
)

print("\n=== 컬럼 역할 분류 ===")
display(classify_df)


=== 컬럼별 채움률 (source 별, 단위 %) ===


,column,biz_fill%,kst_fill%,youth_fill%
0,source,100.0,100.0,100.0
1,source_id,100.0,100.0,100.0
2,title,100.0,100.0,100.0
3,summary,100.0,100.0,100.0
4,s_category,100.0,100.0,100.0
5,provider,100.0,100.0,100.0
6,region,100.0,100.0,100.0
7,raw_target_group,100.0,100.0,100.0
8,raw_support_type,100.0,100.0,100.0
9,raw_target_age,0.0,100.0,100.0



=== 컬럼 역할 분류 ===


,column,역할
0,source,식별자
1,source_id,식별자
2,title,vector text 재료
3,summary,vector text 재료
4,s_category,공통 필터
5,provider,공통 필터
6,region,공통 필터
7,raw_target_group,vector text 재료
8,raw_support_type,vector text 재료
9,raw_target_age,vector text 재료


## 13-3. vector text 빌더 + 메타 정보 저장

**큰 부분**
- main 한 행을 받아 vector DB 에 넣을 **한 줄짜리 텍스트**로 만드는 함수를 정의합니다.
- 룰 버전 / source 별 행수 등 메타 정보는 **별도 json 파일**(`main_v1_1_4.meta.json`)에 한 번만 저장합니다.

**세세한 부분**
- 빌더 함수는 외부 상태에 의존하지 않는 순수 함수로 만들어, Next.js 백엔드 / 인덱싱 스크립트에서도 그대로 재사용 가능.
- 메타 파일은 작고 가볍게 (룰 버전 + 행수 + source 분포만). 매 행 컬럼으로 박지 않습니다.
- 채움률이 낮은 `raw_*` 값은 자동으로 건너뛰므로, 컬럼이 없는 source 라도 안전합니다.


In [70]:
# ======================================================================
# 13-3. vector text 빌더
# ======================================================================
# 큰 부분 :
#   main 한 행을 받아 vector DB 에 적재할 한 줄짜리 텍스트로 만듭니다.
#   supabase 의 pgvector 확장이나 다른 vector DB 모두에 그대로 사용할 수 있습니다.
#
# 세세한 부분 :
#   - 함수는 "외부 상태에 의존하지 않는 순수 함수" 로 만들어,
#     Next.js 백엔드 / 인덱싱 스크립트에서도 그대로 import 해서 쓸 수 있습니다.
#   - 채움률이 낮은 raw_* 컬럼도 후보에 포함하지만, 빈 값은 자동으로 건너뜁니다.
#   - 라벨(예: "[제목]") 형태로 합쳐서 어떤 부분이 무슨 의미인지를
#     vector 모델이 학습하기 쉽게 만듭니다.

# vector text 에 들어갈 컬럼과 그 라벨 (순서대로 합쳐집니다)
VECTOR_TEXT_LABELS = [
    ("title",                     "제목"),
    ("summary",                   "요약"),
    ("s_category",                "분류"),
    ("provider",                  "기관"),
    ("region",                    "지역"),
    ("raw_target_group",          "대상"),
    ("raw_support_type",          "지원형태"),
    ("raw_period_text",           "기간"),
    ("raw_target_age",            "연령"),
    ("raw_income_condition",      "소득조건"),
    ("raw_startup_stage",         "창업단계"),
    ("raw_additional_conditions", "부가조건"),
    ("raw_required_documents",    "서류"),
    ("raw_application_method",    "신청방법"),
]


def build_vector_text(row) -> str:
    """
    main csv 한 행(row, dict 또는 pandas Series) 을 받아
    vector DB 에 적재할 한 줄짜리 텍스트로 만듭니다.

    - 빈 값 / 공백만 있는 값은 자동으로 건너뜁니다.
    - 결과 형식: "[제목] OOO [요약] OOO [분류] OOO ..."
    """
    parts = []
    for col, label in VECTOR_TEXT_LABELS:
        # row 가 dict 든 Series 든 동일하게 동작하게 처리
        if isinstance(row, dict):
            v = row.get(col, "")
        else:
            v = row[col] if col in row else ""
        v = str(v).strip()
        if not v:
            continue   # 빈 값은 건너뜀
        parts.append(f"[{label}] {v}")
    return " ".join(parts)


# ----------------------------------------------------------------------
# vector text 샘플 미리보기 (상위 5건)
# ----------------------------------------------------------------------
# 큰 부분 : 실제로 어떤 텍스트가 만들어지는지 눈으로 한 번 확인합니다.
# 세세한 부분 : 너무 길면 vector 임베딩 비용이 커지니, 길이도 같이 봅니다.

print("=== vector text 샘플 (상위 5건) ===")
for _, r in main_df_review.head(5).iterrows():
    text = build_vector_text(r)
    print("-" * 70)
    print(f"({r['source']}) {r['title'][:40]}...")
    print(f"길이 {len(text)}자")
    print(text[:300] + ("..." if len(text) > 300 else ""))


# ======================================================================
# 13-4. 메타 정보 저장 (main_v1_1_4.meta.json)
# ======================================================================
# 큰 부분 :
#   매 행에 룰 버전 컬럼을 박지 않고, 메타 파일 1개에 한 번만 기록합니다.
#
# 세세한 부분 :
#   - 어떤 룰 csv 를 사용했는지
#   - source 별로 몇 건이 들어갔는지
#   - 정도만 가볍게 저장합니다.
#   - 이 메타 파일은 supabase 적재 / 검토자 전달 시 같이 공유합니다.

META_PATH = CSV_MAIN_DIR / f"main_{VERSION}.meta.json"

meta_info = {
    "main_version": VERSION,
    "category_rule": CAT_RULE_CSV.name,
    "scope_rule": SCOPE_RULE_CSV.name,
    "row_count": int(len(main_df_review)),
    "sources": {s: int((main_df_review["source"] == s).sum()) for s in sources_in_data},
    "generated_by": "PolicyRec_v1_1_4.ipynb",
    "note": "200건 PoC 기준. 룰은 별도 csv 로 관리. supabase 적재용 기준표.",
}

# json 으로 저장 (한글 깨지지 않게 ensure_ascii=False)
META_PATH.write_text(
    json.dumps(meta_info, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n=== 메타 정보 저장 완료 ===")
print("저장 경로:", META_PATH)
print(json.dumps(meta_info, ensure_ascii=False, indent=2))


=== vector text 샘플 (상위 5건) ===
----------------------------------------------------------------------
(biz) 2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고...
길이 760자
[제목] 2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고 [요약] <p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제를 다음과 같이 공고하오니 해당 기술개발과제를 수행하고자 하는 자는 관련 규정에 따라 신청하여 주시기 바랍니다.</p><p><br></p><p style="line-height: 1.8;">☞ 주관연구개발기관 및 공동연구개발기관</p><p style="line-height: 1.8;">- 기업, 대학, 연구기관, 연구조합, 사업자단체, 의료기관 등 국가연구개발혁신...
----------------------------------------------------------------------
(biz) [경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고...
길이 637자
[제목] [경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고 [요약] <p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련 중소기업 지원 확대를 통한 지역 소재부품산업의 경쟁력 강화를 위하여, 「2026년도 경남 소재부품 성장 잠재기업 육성사업」을 아래와 같이 공고하오니, 지원을 희망하시는 관련 기업에서는 많은 참여바랍니다.</p><p><br></p><p>☞ 동부경남지역(양산, 창원, 김해, 밀양 등) 소재부품산업분야 중소기업</p><p><br></p><p style="line-he...
----------------------------------------------------------------------
(biz) [경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라

## 13-5. 인수인계


### (1) 노트북 정제

> 이 노트북은 **200건 기준 PoC**
> - 흐름 / 규칙 / 컬럼 의미는 고정
> - 모듈화 · 확장 · 자동화 필요
> - 핵심 로직은 `app/norm.py`, `app/collectors/*` 확인
> - `_dedupe_key = norm_title + norm_provider + norm_period` 이며 **자동 병합 금지**
> - 컬럼 접두어 규칙 (`s_*`, `raw_*`, `_*`, `norm_*`) 은 그대로 유지하기
> - 대량 수집은 `app/collectors/youth.py` 에 추가된 `pageNum` / `pageSize` 파라미터로 페이지 루프만 돌리면 됨
> - 룰 버전 추적은 별도 csv (`cat_rule_v1_1_4.csv`, `scope_rule_v1_1_4.csv`) + `main_v1_1_4.meta.json` 으로 관리. **행 컬럼으로는 박지 않습니다.**

### (2) DB (supabase)

> CSV 는 사람 검토용 기준표입니다. 카드 / 상세 / 챗봇은 CSV 를 직접 읽지 않습니다.
>
> - **입력 파일** : `data/csv/main/main_v1_1_4.csv` (568행 × 25컬럼) + `main_v1_1_4.meta.json`
> - **PK 후보** : `(source, source_id)` 복합키 또는 supabase 자동 id
> - **메인 필터로 쓸 컬럼** : `s_category`, `provider`, `region` (3개 source 모두 잘 채워짐)
> - **메인 필터로 쓰지 말 컬럼** : 채움률 편중 큰 `raw_*` 컬럼들 (예: `raw_income_condition` 은 youth 전용)
> - **vector text** : 노트북의 `build_vector_text(row)` 함수로 생성 → supabase `pgvector` 확장 추천
> - **링크** : `norm_detail_url` (정규화) / 원문은 `raw_detail_url`

### (3) 앱 (Next.js)

> 앱은 streamlit 이 아니라 **Next.js** 로 갈 예정. `streamlit_app3*.py` 는 PoC 참고용
>
> - Next.js → supabase JS client 직접 호출 권장 (별도 API 라우트 거의 불필요)
> - **카드 / 상세 화면 표시 컬럼** : `title`, `summary`, `s_category`, `provider`, `region`, `raw_period_text`, `norm_detail_url`
> - **검색** : vector 컬럼 (`build_vector_text` 결과 임베딩) 기반
> - **필터 UI** : `s_category`, `provider`, `region` 3개로만 구성 (sparse 한 raw_* 컬럼은 노출 X)

### 알려진 한계

- 200건 PoC 기준 (대량 수집 / 페이지네이션은 다음 단계)
- dedupe 는 강한 신호 (URL exact match) 만 자동, 약한 신호는 후보로만 표시 (자동 병합 X)
- youth source 는 `norm_provider` 채움률이 약 16% → 별도 보강 매핑이 필요할 수 있음
- 모든 `raw_*` 컬럼이 모든 source 를 커버하지는 않음 (채움률 표 참고)
